In [1]:
print("hello world")

hello world


In [13]:
import os
# Включаем совместимость с MPS (спасает от кастомных слоев на Apple Silicon)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from ultralytics import YOLO

# Жестко привязываемся к текущей папке проекта
CURRENT_DIR = os.getcwd()

# 1. Загружаем предобученные веса для ДЕТЕКЦИИ
model = YOLO("yolo11s.pt") 

# 👉 ШАГ 1: Обучение модели детекции опухолей на чипе M4 (MPS)
print("🏋️‍♂️ Начинаем дообучение на графическом процессоре M4 (MPS)...")
results = model.train(
    data="../models/axial_t1wce_2_class/axial_t1wce_2_class.yaml", # Путь к yaml-файлу
    epochs=50,                  
    imgsz=640,                  
    batch=16,                   
    device="mps",               
    workers=4,
    cache=False,
    val=False,                  # Выключаем встроенный val, чтобы не крашнуть ядро на MPS

    # 🟢 ИДЕАЛЬНАЯ АУГМЕНТАЦИЯ ДЛЯ МРТ (Медицинский конфиг)
    degrees=15.0,               # Небольшие повороты (голова пациента может лежать неровно)
    translate=0.1,              # Легкий сдвиг по осям (кадрирование снимка)
    scale=0.15,                 # Небольшой зум (масштаб опухоли к размеру мозга)
    shear=0.0,                  # Отключаем сдвиг параллелограммом (искажает анатомию)
    perspective=0.0,            # Отключаем 3D-перспективу (МРТ — это плоский 2D-срез)
    
    fliplr=0.5,                 # Отражение лево-право (мозг симметричен, удваивает данные)
    flipud=0.0,                 # СТРОГО 0.0: Переворот верх-низ ломает анатомическую ориентацию
    
    mosaic=0.0,                 # СТРОГО 0.0: Склейка 4 снимков в один нереалистична для МРТ
    mixup=0.0,                  # Отключаем смешивание слоев (создает "фантомные" опухоли)
    copy_paste=0.0,             # Отключаем копирование объектов на другие снимки

    # Изменение яркости/интенсивности серого (разные томографы)
    hsv_h=0.0,                  # На МРТ нет цвета (оттенка), фиксируем в 0
    hsv_s=0.0,                  # На МРТ нет насыщенности цвета, фиксируем в 0
    hsv_v=0.15,                 # Случайное изменение яркости оттенков серого на ±15%
    
    # ❌ Удалены параметры 'blur' и 'median', которые вызывали SyntaxError

    # Настройки оптимизатора
    lr0=0.01, 
    lrf=0.01, 
    momentum=0.937, 
    weight_decay=0.0005,

    # Сохранение результатов в папку вашего проекта
    project=os.path.join(CURRENT_DIR, "models"), 
    name="yolo11s_tumor_detection"
)

# 👉 ШАГ 2: Перенос валидации на CPU (защита от вылета NMS-слоев на macOS)
print("\n--- 🎉 Обучение завершено. Запуск безопасной валидации на CPU ---")

# Важно: загружаем лучшую сохраненную модель перед валидацией
best_model_path = os.path.join(CURRENT_DIR, "models", "yolo11s_tumor_detection", "weights", "best.pt")
validation_model = YOLO(best_model_path)

metrics = validation_model.val(
    data="../models/axial_t1wce_2_class/axial_t1wce_2_class.yaml",
    device="cpu",               # Валидация строго на процессоре
    imgsz=640,
    workers=4,
    project=os.path.join(CURRENT_DIR, "models"),
    name="yolo11s_tumor_val"
)

print(f"\n✅ Всё готово! Валидация завершена успешно.")
print(f"Лучшие веса сохранены здесь: {best_model_path}")

🏋️‍♂️ Начинаем дообучение на графическом процессоре M4 (MPS)...
Ultralytics 8.4.60 🚀 Python-3.13.13 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../models/axial_t1wce_2_class/axial_t1wce_2_class.yaml, degrees=15.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.15, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolo11s_tumor_detection-3, nbs=64

KeyboardInterrupt: 

In [2]:
import os
from ultralytics import YOLO

# 1. Автоматически определяем пути на основе вашей структуры проекта
CURRENT_DIR = os.getcwd()  # Мы находимся в папке notebooks

# Поднимаемся на уровень выше и идем к весам
# Внимание: Проверьте точное имя папки 'yolo11m_tumor_detection_b...' и подставьте его ниже, если оно отличается
MODEL_DIR = os.path.join(
    os.path.dirname(CURRENT_DIR), 
    "models", 
    "yolo11m_tumor_detection_best_config-2"  # 👈 Замените три точки на точное имя папки, если оно обрезано
)
BEST_WEIGHTS = os.path.join(MODEL_DIR, "weights", "best.pt")
DATA_YAML = os.path.join(os.path.dirname(CURRENT_DIR), "models", "axial_t1wce_2_class", "axial_t1wce_2_class.yaml")

# 2. Загружаем обученную модель
print(f"📦 Загружаем модель: {BEST_WEIGHTS}")
model = YOLO(BEST_WEIGHTS)

# --- ШАГ 1: Метрики на валидационной выборке (Split / Val) ---
print("\n📊 [1/2] Расчет метрик на ВАЛИДАЦИОННОЙ выборке (Split)...")
val_metrics = model.val(
    data=DATA_YAML,
    split="val",     # Явно указываем сплит валидации
    device="cpu",    # Безопасно для Mac M4
    imgsz=640,
    workers=4,
    project=MODEL_DIR,
    name="metrics_val_split"
)

# --- ШАГ 2: Метрики на обучающей выборке (Train) ---
print("\n📈 [2/2] Расчет метрик на ОБУЧАЮЩЕЙ выборке (Train)...")
train_metrics = model.val(
    data=DATA_YAML,
    split="train",   # Переключаем сплит на тренировочный
    device="cpu",    # Безопасно для Mac M4
    imgsz=640,
    workers=4,
    project=MODEL_DIR,
    name="metrics_train_split"
)

# --- ВЫВОД РЕЗУЛЬТАТОВ В КОНСОЛЬ ---
print("\n" + "="*50)
print("🏆 ИТОГОВЫЕ МЕТРИКИ (mAP50 и mAP50-95):")
print("="*50)
print(f"VAL   -> mAP50: {val_metrics.box.map50:.4f} | mAP50-95: {val_metrics.box.map:.4f}")
print(f"TRAIN -> mAP50: {train_metrics.box.map50:.4f} | mAP50-95: {train_metrics.box.map:.4f}")
print("="*50)
print(f"📝 Подробные отчеты и новые графики сохранены в папку:\n{MODEL_DIR}")

📦 Загружаем модель: /Users/romanrahmatulin/Projects/phasa3/segmentproject/models/yolo11m_tumor_detection_best_config-2/weights/best.pt

📊 [1/2] Расчет метрик на ВАЛИДАЦИОННОЙ выборке (Split)...
Ultralytics 8.4.60 🚀 Python-3.13.13 torch-2.12.0 CPU (Apple M4)
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 63.1±25.2 MB/s, size: 9.7 KB)
val: Scanning /Users/romanrahmatulin/Projects/phasa3/segmentproject/models/axial_t1wce_2_class/labels/test... 75 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75 4.6Kit/s 0.0s
val: New cache created: /Users/romanrahmatulin/Projects/phasa3/segmentproject/models/axial_t1wce_2_class/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 7.9s/it 39.6s<11.8s
                   all         75         77      0.914      0.819      0.903      0.624
              negative         40         40    

In [4]:
import sys

def check_library(lib_name):
    try:
        __import__(lib_name)
        # Для Pillow проверяется PIL, но выводим красивое имя
        display_name = "Pillow (PIL)" if lib_name == "PIL" else lib_name
        print(f"✅ {display_name:<15} — УСТАНОВЛЕНА")
        return True
    except ImportError:
        display_name = "Pillow (PIL)" if lib_name == "PIL" else lib_name
        print(f"❌ {display_name:<15} — НЕ НАЙДЕНА")
        return False

print(f"🐍 Текущее окружение Python: {sys.executable}\n")
print("🔍 Проверка необходимых библиотек:")
print("-" * 40)

required_libs = ["ultralytics", "streamlit", "requests", "PIL"]
results = [check_library(lib) for lib in required_libs]
print("-" * 40)

if all(results):
    print("🚀 Отлично! Все библиотеки на месте, можно запускать Streamlit-приложение.")
else:
    print("⚠️ Некоторые библиотеки отсутствуют. Чтобы их установить, выполните команду ниже.")

🐍 Текущее окружение Python: /Users/romanrahmatulin/Projects/phasa3/segmentproject/env/bin/python

🔍 Проверка необходимых библиотек:
----------------------------------------
✅ ultralytics     — УСТАНОВЛЕНА
✅ streamlit       — УСТАНОВЛЕНА
✅ requests        — УСТАНОВЛЕНА
✅ Pillow (PIL)    — УСТАНОВЛЕНА
----------------------------------------
🚀 Отлично! Все библиотеки на месте, можно запускать Streamlit-приложение.
